# Q2 - NoSQL Database Implementation (MongoDB)
## Big Data Analytical Methods - CA2


This notebook demonstrates the use of MongoDB as a NoSQL database to store and query 
the Global Weather Repository dataset. I am loading the processed data from HDFS 
(via PySpark) and inserting it into a MongoDB collection, then performing three 
queries to extract meaningful information from the database.

**Database:** MongoDB 7.0  
**Collection:** weather_data  
**Connection:** mongodb://localhost:27017/

## Step 1: Connect to MongoDB

In this step I am establishing a connection to the local MongoDB instance running 
in WSL. I am creating a new database called "weather_db" and a collection called 
"weather_data" where I will store the weather records.

In [1]:
from pymongo import MongoClient

# Connect to local MongoDB instance
client = MongoClient("mongodb://localhost:27017/")

# Create/connect to database and collection
db = client["weather_db"]
collection = db["weather_data"]

# Confirm connection
print("Connected to MongoDB successfully")
print("Database:", db.name)
print("Collection:", collection.name)

Connected to MongoDB successfully
Database: weather_db
Collection: weather_data


## Step 2: Load Data from HDFS using PySpark

In this step I am using PySpark to read the cleaned weather dataset which I previously
processed in Q1. I am then converting the Spark DataFrame into a format that MongoDB 
can accept (a list of dictionaries) so that it can be inserted into the collection.

In [2]:
import os
os.environ["JAVA_HOME"] = r"C:\Program Files\Microsoft\jdk-11.0.31.11-hotspot"

from pyspark.sql import SparkSession

# Create Spark Session
spark = SparkSession.builder \
    .appName("WeatherMongoDB") \
    .master("local[*]") \
    .getOrCreate()

# Load dataset
df = spark.read.csv(
    r"C:\Users\arkje\OneDrive\Documents\GIT\big-data-weather-analytics\DATASETS\GlobalWeatherRepository.csv",
    header=True,
    inferSchema=True
)

# Clean data
df_clean = df.dropDuplicates().dropna(subset=["country", "temperature_celsius", "humidity"])

print("Rows loaded:", df_clean.count())
print("Spark session ready")

Rows loaded: 148515
Spark session ready


## Step 3: Insert Data into MongoDB

In this step I am converting the Spark DataFrame to a list of dictionaries 
and inserting the records into the MongoDB weather_data collection.
I am inserting a sample of 10,000 records to keep the process efficient 
while still demonstrating the full pipeline.

In [3]:
# Convert Spark DataFrame to list of dictionaries for MongoDB
# Using a sample of 10,000 records for efficient insertion
df_sample = df_clean.limit(10000)

# Convert to pandas then to dictionary records
records = df_sample.toPandas().to_dict("records")

# Clear existing collection to avoid duplicates
collection.drop()
collection = db["weather_data"]

# Insert records into MongoDB
result = collection.insert_many(records)

print("Records inserted:", len(result.inserted_ids))
print("Data successfully stored in MongoDB")

Records inserted: 10000
Data successfully stored in MongoDB


## Step 4: NoSQL Queries

In this step I am performing three queries on the MongoDB weather_data collection
to extract meaningful information from the stored weather data.
These queries demonstrate the flexibility and power of NoSQL querying capabilities.

### Query 1 - Find all records where temperature is above 40°C

In this query I am retrieving all weather records where the temperature 
exceeds 40 degrees Celsius. This helps identify the hottest locations 
in the dataset and demonstrates basic filtering in MongoDB.

In [4]:
# Query 1: Find all records where temperature is above 40°C
print("Q2: Figure 4 - Records with Temperature above 40°C:")
print("-" * 60)

results = collection.find(
    {"temperature_celsius": {"$gt": 40}},
    {"_id": 0, "country": 1, "location_name": 1, "temperature_celsius": 1, "condition_text": 1}
)

count = 0
for record in results:
    print(record)
    count += 1

print("-" * 60)
print(f"Total records found: {count}")

Q2: Figure 4 - Records with Temperature above 40°C:
------------------------------------------------------------
{'country': 'India', 'location_name': 'New Delhi', 'temperature_celsius': 44.8, 'condition_text': 'Clear'}
{'country': 'Kuwait', 'location_name': 'Kuwait City', 'temperature_celsius': 47.6, 'condition_text': 'Sunny'}
{'country': 'Iraq', 'location_name': 'Baghdad', 'temperature_celsius': 44.4, 'condition_text': 'Sunny'}
{'country': 'Saudi Arabia', 'location_name': 'Riyadh', 'temperature_celsius': 41.0, 'condition_text': 'Partly cloudy'}
{'country': 'India', 'location_name': 'New Delhi', 'temperature_celsius': 43.0, 'condition_text': 'Mist'}
{'country': 'India', 'location_name': 'New Delhi', 'temperature_celsius': 41.0, 'condition_text': 'Mist'}
{'country': 'Iraq', 'location_name': 'Baghdad', 'temperature_celsius': 41.2, 'condition_text': 'Sunny'}
{'country': 'Kuwait', 'location_name': 'Kuwait City', 'temperature_celsius': 47.2, 'condition_text': 'Sunny'}
{'country': 'Iraq', '

### Query 2 - Average humidity grouped by country

In this query I am using MongoDB aggregation pipeline to calculate the average 
humidity for each country and sorting the results in descending order.
This demonstrates MongoDB's ability to perform group-by operations similar 
to SQL GROUP BY but using NoSQL aggregation framework.

In [5]:
# Query 2: Average humidity by country using aggregation pipeline
print("Q2: Figure 5 - Average Humidity by Country (Top 15):")
print("-" * 50)

pipeline = [
    {"$group": {
        "_id": "$country",
        "avg_humidity": {"$avg": "$humidity"}
    }},
    {"$sort": {"avg_humidity": -1}},
    {"$limit": 15},
    {"$project": {
        "_id": 0,
        "country": "$_id",
        "avg_humidity": {"$round": ["$avg_humidity", 2]}
    }}
]

results = collection.aggregate(pipeline)

for record in results:
    print(f"Country: {record['country']:<30} Avg Humidity: {record['avg_humidity']}%")

print("-" * 50)

Q2: Figure 5 - Average Humidity by Country (Top 15):
--------------------------------------------------
Country: Belize                         Avg Humidity: 95.69%
Country: Letonia                        Avg Humidity: 94.0%
Country: Dominican Republic             Avg Humidity: 93.33%
Country: Suriname                       Avg Humidity: 93.11%
Country: Guyana                         Avg Humidity: 93.08%
Country: Cote d'Ivoire                  Avg Humidity: 91.92%
Country: Cuba                           Avg Humidity: 91.05%
Country: Panama                         Avg Humidity: 90.91%
Country: Guatemala                      Avg Humidity: 90.82%
Country: Fiji Islands                   Avg Humidity: 88.72%
Country: United States of America       Avg Humidity: 87.76%
Country: El Salvador                    Avg Humidity: 87.42%
Country: Samoa                          Avg Humidity: 87.12%
Country: Trinidad and Tobago            Avg Humidity: 86.81%
Country: Equatorial Guinea              Avg

### Query 3 - Find countries with high wind speed and poor air quality

In this query I am combining two conditions to find records where wind speed 
is above 50 kph AND PM2.5 air quality is above 50 μg/m³.
This demonstrates MongoDB's ability to filter using multiple conditions 
simultaneously which is useful for identifying extreme weather events.

In [13]:
# Query 3: High wind speed AND high UV index
print("Q2: Figure 6 - Records with High Wind Speed (>40 kph) AND High UV Index (>8):")
print("-" * 70)

results = collection.find(
    {
        "wind_kph": {"$gt": 40},
        "uv_index": {"$gt": 8}
    },
    {
        "_id": 0,
        "country": 1,
        "location_name": 1,
        "wind_kph": 1,
        "uv_index": 1,
        "condition_text": 1
    }
).limit(15)

count = 0
for record in results:
    print(f"Country: {record['country']:<25} Location: {record['location_name']:<20} "
          f"Wind: {record['wind_kph']} kph  UV Index: {record['uv_index']}")
    count += 1

print("-" * 70)
print(f"Records shown: {count}")

Q2: Figure 6 - Records with High Wind Speed (>40 kph) AND High UV Index (>8):
----------------------------------------------------------------------
Country: Turkey                    Location: Yaren                Wind: 43.9 kph  UV Index: 9.0
Country: Kuwait                    Location: Kuwait City          Wind: 43.6 kph  UV Index: 10.2
Country: Azerbaijan                Location: Baku                 Wind: 54.7 kph  UV Index: 9.9
----------------------------------------------------------------------
Records shown: 3


## Step 5: Scalability Advantages of NoSQL Databases

When comparing MongoDB to traditional relational databases such as MySQL or PostgreSQL,
there are several key scalability advantages worth discussing.

Relational databases scale vertically, meaning to handle more data you need to upgrade
the hardware of a single server which is expensive and has physical limits.
MongoDB on the other hand scales horizontally through a process called sharding,
where data is distributed across multiple servers or nodes. This means that as the 
dataset grows, more servers can simply be added to the cluster without downtime.

In the context of this project, the Global Weather Repository contains data from 
hundreds of countries and could potentially grow to millions of records if updated 
continuously. A relational database would struggle with this scale, while MongoDB 
can distribute the collection across multiple shards and handle the load efficiently.

Additionally, MongoDB uses a flexible document model which means that new fields 
such as additional weather sensors or new air quality metrics can be added to 
documents without altering the entire database schema, unlike SQL which requires 
ALTER TABLE operations that can lock tables and cause downtime.

In [ ]:
## Step 6: Data Privacy and Security Considerations

When storing weather data in MongoDB there are several important security 
considerations to keep in mind.

First, in this project I have MongoDB running without authentication enabled 
which is acceptable for a local development environment but would be completely 
unacceptable in a production system. In a real deployment, authentication should 
be enabled and each user should have role-based access control (RBAC) configured 
so that only authorised users can read or write data.

Second, if the database contained personal data such as user locations or device 
identifiers from weather sensors, encryption at rest and in transit would be 
required. MongoDB supports TLS/SSL encryption for data in transit and encrypted 
storage engines for data at rest.

Third, regular backups should be configured using MongoDB's built-in tools such 
as mongodump to prevent data loss. In a production environment, replica sets 
should be used to ensure high availability and automatic failover.

Finally, network access should be restricted using firewall rules so that only 
trusted IP addresses can connect to the MongoDB instance, and the default port 
27017 should be changed to reduce the risk of automated attacks.

## Step 7: Ethical Issues Related to the Use of Personal Data

Although the Global Weather Repository dataset does not contain directly personal 
data, there are still important ethical considerations when working with large 
scale datasets in general.

If weather data were collected from personal devices such as smartphones or 
smart home sensors, it could potentially reveal information about a person's 
location, daily routine and behaviour patterns. This type of data would fall 
under the General Data Protection Regulation (GDPR) in the European Union and 
would require explicit consent from users before collection and processing.

In the context of this project, the data was sourced from a public repository 
on Kaggle and contains only aggregated weather observations without any 
personally identifiable information (PII). However, if this system were 
deployed in a real smart city environment, the following ethical principles 
should be followed:

- **Data minimisation** - only collect data that is strictly necessary
- **Purpose limitation** - data collected for weather analysis should not 
  be used for other purposes without consent
- **Transparency** - users should be informed about what data is collected 
  and how it is used
- **Right to erasure** - users should have the right to request deletion 
  of their data at any time

These principles ensure that big data systems are used responsibly and in 
compliance with data protection laws.

## Summary of Q2 Analysis

In this notebook I successfully completed the following tasks:

1. **MongoDB Connection** - Connected to a local MongoDB 7.0 instance running 
   in WSL2 on Ubuntu 22.04.

2. **Data Insertion** - Loaded 10,000 weather records from the Global Weather 
   Repository dataset into a MongoDB collection called weather_data.

3. **Query 1** - Retrieved all records with temperature above 40°C, 
   finding 89 records mostly from Middle Eastern countries.

4. **Query 2** - Used aggregation pipeline to calculate average humidity 
   by country, identifying Belize as the most humid country with 95.69%.

5. **Query 3** - Combined multiple conditions to find records with high 
   wind speed (>40 kph) and high UV index (>8), demonstrating MongoDB's 
   flexible querying capabilities.

6. **Discussion** - Covered scalability advantages of NoSQL over relational 
   databases, data privacy and security considerations, and ethical issues 
   related to personal data usage.